In [1]:
from newsapi import NewsApiClient
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install setuptools
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews
from tqdm import tqdm
import requests
import json

import warnings
warnings.filterwarnings('ignore')


# Get S&P500 companies' tickers

In [2]:
response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
companies_info_df = pd.read_html(response.content)[0]
companies_info_df = companies_info_df.drop(columns=["No.", "Stock Price", "% Change", "Revenue"])
companies_info_df.to_excel("Data/SPY_companies_info.xlsx", index=False)

In [3]:
# tech stocks, pharma stocks, oil stocks, tobacco stocks and Market
tickers = companies_info_df["Symbol"].to_list()

# News Data Download

## CapIQ Data
Data is acquired from SMU Library Website https://researchguides.smu.edu.sg/az.php?a=c

In [4]:
capiq_df = pd.read_excel("Data/capiq_news_data.xlsx")
capiq_df = capiq_df.rename(columns = {"Key Developments By Date": "date",
                            "Key Development Headline": "title",
                            "Key Development Sources": "source",
                            "Primary Industry": "topic"
                            })

capiq_df =  capiq_df.drop(columns=["Key Developments by Type", "Key Development Situation"])

In [5]:
capiq_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448274 entries, 0 to 448273
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   date                  448274 non-null  datetime64[ns]
 1   Company Name(s)       448274 non-null  object        
 2   title                 448274 non-null  object        
 3   source                448274 non-null  object        
 4   Business Description  448274 non-null  object        
 5   topic                 448274 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 20.5+ MB


In [15]:
# Get ticker
capiq_df["Ticker"] = capiq_df["Company Name(s)"].str.replace(r'[^(]*\(|\)[^)]*', '')
capiq_df["Ticker"] = capiq_df["Ticker"].str.split(':').str[-1]
capiq_df["Ticker"] = capiq_df["Ticker"].str.replace(r'[^a-zA-Z]+', '')
capiq_df["Company Name(s) - Cleaned"] = capiq_df["Company Name(s)"].str.split("(").str[0]

print(capiq_df.shape)
capiq_df

(448274, 8)


,date,Company Name(s),title,source,Business Description,topic,Ticker,Company Name(s) - Cleaned
0,2014-01-01,"Motorola Solutions, Inc. (NYSE:MSI)",Motorola Solutions to Provide IDF's Battlefiel...,Other,"Motorola Solutions, Inc. provides public safet...",Communications Equipment,MSI,"Motorola Solutions, Inc."
1,2014-01-01,TE Connectivity plc (NYSE:TEL),TE Connectivity Brings Advanced Mobile Service...,Business Wire,"TE Connectivity plc, together with its subsidi...",Electronic Manufacturing Services,TEL,TE Connectivity plc
2,2014-01-01,"Leidos Holdings, Inc. (NYSE:LDOS)",Leidos Holdings Receives Follow-On Contract fr...,Datamonitor NewsWire,"Leidos Holdings, Inc., together with its subsi...",Research and Consulting Services,LDOS,"Leidos Holdings, Inc."
3,2014-01-01,MarketAxess Holdings Inc. (NasdaqGS:MKTX),MarketAxess Holdings Inc.'s Equity Buyback ann...,Capital IQ Buybacks Database,"MarketAxess Holdings Inc., together with its s...",Financial Exchanges and Data,MKTX,MarketAxess Holdings Inc.
4,2014-01-01,Amgen Inc. (NasdaqGS:AMGN); UCB SA (ENXTBR:UCB),Amgen and UCB Announces Results from Phase 2 T...,PR Newswire,Amgen Inc. (NasdaqGS:AMGN)Amgen Inc. discovers...,Amgen Inc. (NasdaqGS:AMGN) (Biotechnology); UC...,UCB,Amgen Inc.
...,...,...,...,...,...,...,...,...
448269,2024-12-04,CSX Corporation (NasdaqGS:CSX),CSX Corporation Presents at UBS Global Industr...,PR Newswire; Business Wire; GlobeNewswire; Com...,"CSX Corporation, together with its subsidiarie...",Rail Transportation,CSX,CSX Corporation
448270,2024-12-04,UnitedHealth Group Incorporated (NYSE:UNH),UnitedHealth Group Incorporated - Analyst/Inve...,Business Wire,UnitedHealth Group Incorporated operates as a ...,Managed Health Care,UNH,UnitedHealth Group Incorporated
448271,2024-12-04,LyondellBasell Industries N.V. (NYSE:LYB),LyondellBasell Industries N.V. Presents at Gol...,PR Newswire; Business Wire; GlobeNewswire; Com...,LyondellBasell Industries N.V. operates as a c...,Commodity Chemicals,LYB,LyondellBasell Industries N.V.
448272,2024-12-04,Freeport-McMoRan Inc. (NYSE:FCX),Freeport-McMoRan Inc. Presents at Mines and Mo...,Company Website,Freeport-McMoRan Inc. engages in the mining of...,Copper,FCX,Freeport-McMoRan Inc.


In [ ]:
capiq_df = capiq_df[capiq_df['Ticker'].isin(tickers)]
print(capiq_df.shape)


(427102, 8)


In [36]:
capiq_df2 = capiq_df.groupby(['Ticker']).first().drop_duplicates()
company_names = capiq_df.groupby(['Ticker']).first().drop_duplicates()["Company Name(s) - Cleaned"].tolist()
tickers_sorted = capiq_df.groupby(['Ticker']).first().drop_duplicates().index.tolist()

len(company_names)

498

## Pygoognews 
https://github.com/kotartemiy/pygooglenews


In [ ]:
# Looping over 500 tickers: this run will take a few hours
gn = GoogleNews(lang = 'en')

news_df = pd.DataFrame()
for i, company in tqdm(enumerate(company_names)):
    top = gn.search(company)
    entries = top["entries"]
    df_temp = clean_goog_news(entries)
    df_temp["date"] = df_temp["date"].apply(lambda d: pd.to_datetime(d, errors='coerce').date())
    df_temp = df_temp.dropna()
    df_temp["Ticker"] = tickers_sorted[i]
    news_df = pd.concat([news_df.copy(), df_temp.copy()])


498it [3:25:05, 24.71s/it]


In [ ]:
news_df

,date,title,source,Ticker
0,2024-12-14,Stifel Financial Corp Trims Stake in Agilent T...,https://news.google.com/rss/articles/CBMivgFBV...,A
1,2024-12-13,Is There Now An Opportunity In Agilent Technol...,https://news.google.com/rss/articles/CBMi5gFBV...,A
2,2024-12-13,Agilent Technologies (A) Announces Reorganizat...,https://news.google.com/rss/articles/CBMi9gFBV...,A
3,2024-12-13,"Metagenomics Market Report by Product, Technol...",https://news.google.com/rss/articles/CBMi7AJBV...,A
4,2024-12-13,Agilent Technologies Announces Operational Res...,https://news.google.com/rss/articles/CBMiswFBV...,A
...,...,...,...,...
95,2024-11-19,Zoetis Inc. (NYSE:ZTS) Shares Sold by Cantillo...,https://news.google.com/rss/articles/CBMixgFBV...,ZTS
96,2024-11-20,Zoetis Inc. (NYSE:ZTS) Holdings Lowered by Por...,https://news.google.com/rss/articles/CBMiswFBV...,ZTS
97,2024-11-26,Bank of Montreal Can Has $191.81 Million Holdi...,https://news.google.com/rss/articles/CBMitwFBV...,ZTS
98,2024-11-21,"First Horizon Advisors Inc. Sells 7,987 Shares...",https://news.google.com/rss/articles/CBMiugFBV...,ZTS


In [47]:
news_df.to_csv("Data/goog_news.csv")

## General News 
- Newscatcher for general news not targetted at specific stocks: https://github.com/kotartemiy/newscatcher


In [48]:
topics = ['tech', 'news', 'business', 'science', 'finance', 'food', 'politics', 'economics', 'travel', 'entertainment', 'music', 'sport', 'world']
all_general_news = pd.DataFrame()

for t in tqdm(topics):
    news_temp = clean_newscatcher_ news(t, n=5)
    all_general_news = pd.concat([all_general_news, news_temp])



  0%|          | 0/13 [00:00<?, ?it/s]

No. of URLs: 163
['facebook.com', 'twitter.com', 'theguardian.com', 'cnet.com', 'wired.com', 'digg.com', 'samsung.com', 'theverge.com', 'mashable.com', 'engadget.com', 'businesswire.com', 'lifehacker.com', 'utoronto.ca', 'eff.org', 'theregister.co.uk', 'pcworld.com', 'prweb.com', 'acm.org', 'techradar.com', 'ndtv.com', 'boingboing.net', 'popularmechanics.com', 'crunchbase.com', 'ycombinator.com', 'macrumors.com', 'cio.com', 'gigaom.com', 'ibtimes.co.uk', 'networkworld.com', 'extremetech.com', 'windows.com', 'dailydot.com', 'gsmarena.com', 'anandtech.com', 'slashgear.com', 'computerweekly.com', 'betanews.com', 'techdirt.com', 'houstonchronicle.com', 'techtimes.com', 'androidcentral.com', 'techspot.com', 'infoq.com', 'macmillan.com', 'windowscentral.com', 'neowin.net', 'phonearena.com', 'tmcnet.com', 'pocket-lint.com', 'cbinsights.com', 'theiet.org', 'thewirecutter.com', 'gizmodo.com.au', 'nerdist.com', 'itworld.com', 'internetsociety.org', 'themarysue.com', 'laptopmag.com', 'wccftech.co

100%|██████████| 13/13 [46:48<00:00, 216.00s/it] 


In [49]:
all_general_news["date"] = all_general_news["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
all_general_news = all_general_news.dropna()
all_general_news.sort_values(by='date', inplace=True)
all_general_news = all_general_news.reset_index(drop=True)

all_general_news

,date,title,source,topic


In [50]:
all_general_news.to_csv("Data/general_news.csv")

## Merging CapIQ, PyGoogleNews and NewsCatcher

- CapIQ and PyGoogleNews will be concatenated based on the 'ticker' column
- CapIQ and NewsCatcher will be concatenated based on the 'topic' column 
    * NewsCatcher has less and defined topics for each news. Hence, 'glove-twitter-25' is used to evaluate which topic for each stock in CapIQ has the closest similarity to the topic in NewsCatcher
    * Note that 'glove-twitter-25' required both texts to have the same number of words to evaluate its similarity, hence 7 random non-NA rows of the first word of the topic for each stock in CapIQ will be evaluated against each topic found in NewsCatcher

In [57]:
capiq_df_tickers = capiq_df["Ticker"].unique().tolist()

capiq_newscatcher_df = pd.DataFrame()
for t in tqdm(capiq_df_tickers):
    news_temp = capiq_df[capiq_df["Ticker"] == t].copy()
    
    match_topic_newscatcher = get_topic(news_temp["Business Description"].dropna()) # already sorted by ticker, just get the first non-NA topic
    curr_newscatcher = all_general_news[all_general_news["topic"].str.contains(match_topic_newscatcher)].copy()
    curr_newscatcher["Ticker"] = t
    capiq_newscatcher_df = pd.concat([capiq_newscatcher_df, news_temp.drop(columns=["Company Name(s)", "Business Description"]).copy(), curr_newscatcher.copy()])
   

100%|██████████| 498/498 [00:54<00:00,  9.07it/s]


In [58]:
print(capiq_newscatcher_df.shape)
capiq_newscatcher_df

,date,title,source,topic,Ticker,Company Name(s) - Cleaned
0,2014-01-01,Motorola Solutions to Provide IDF's Battlefiel...,Other,Communications Equipment,MSI,"Motorola Solutions, Inc."
21,2014-01-02,"Motorola Solutions, Inc. (NYSE:MSI) acquired T...",Capital IQ Transaction Database,Chart Venture Partners (Asset Management and C...,MSI,Chart Venture Partners; Core Management II Cor...
1114,2014-01-16,"City of Charlotte, N.C. Works with Motorola So...",Business Wire,Communications Equipment,MSI,"Motorola Solutions, Inc."
1409,2014-01-21,"Motorola Solutions, Inc. to Report Q4, 2013 Re...",Business Wire,Communications Equipment,MSI,"Motorola Solutions, Inc."
1513,2014-01-22,"Motorola Solutions, Inc., Q4 2013 Earnings Cal...",Business Wire; Company Website,Communications Equipment,MSI,"Motorola Solutions, Inc."
...,...,...,...,...,...,...
443031,2024-10-23,"GE Vernova Inc., Q3 2024 Earnings Call, Oct 23...",Company Website,Heavy Electrical Equipment,GEV,GE Vernova Inc.
443280,2024-10-24,GE Vernova Inc. expected to report Q3 2024 res...,Capital IQ Expected Earnings Database,Heavy Electrical Equipment,GEV,GE Vernova Inc.
445387,2024-11-04,"GE Vernova Inc. Presents at ADIPEC 2024, Nov-0...",Company Website; PR Newswire Europe,Heavy Electrical Equipment,GEV,GE Vernova Inc.
446594,2024-11-13,GE Vernova Inc. Presents at Innovation Summit ...,PR Newswire; Company Website,Heavy Electrical Equipment,GEV,GE Vernova Inc.


In [62]:
all_news_df = pd.concat([capiq_newscatcher_df, news_df])
all_news_df["date"] = all_news_df["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
all_news_df = all_news_df.dropna(subset=["date", "Ticker", "title"])
all_news_df.sort_values(by='date', inplace=True)
all_news_df = all_news_df.reset_index(drop=True)

all_news_df

,date,title,source,topic,Ticker,Company Name(s) - Cleaned
0,2008-02-01,Bell Company and DRA Advisors Announce Plans t...,https://news.google.com/rss/articles/CBMixwFBV...,NaN,UDR,NaN
1,2008-05-15,Boulder to be first “Smart Grid City” - POWER ...,https://news.google.com/rss/articles/CBMibkFVX...,NaN,XEL,NaN
2,2008-05-15,Howard Hughes: A revolutionary recluse - Las V...,https://news.google.com/rss/articles/CBMiekFVX...,NaN,UHS,NaN
3,2009-09-08,Xcel launches Boulder as first 'smart grid' ci...,https://news.google.com/rss/articles/CBMidEFVX...,NaN,XEL,NaN
4,2011-11-12,Calvert Renovation to Begin in January - Patch,https://news.google.com/rss/articles/CBMifEFVX...,NaN,UDR,NaN
...,...,...,...,...,...,...
472244,2024-12-14,"Tidal Investments LLC Sells 67,257 Shares of I...",https://news.google.com/rss/articles/CBMiuAFBV...,NaN,INTC,NaN
472245,2024-12-14,Ayar Labs Closes a $155M Series D from AMD Int...,https://news.google.com/rss/articles/CBMimAFBV...,NaN,INTC,NaN
472246,2024-12-14,Parker-Hannifin Co. (NYSE:PH) Shares Acquired ...,https://news.google.com/rss/articles/CBMivgFBV...,NaN,PH,NaN
472247,2024-12-14,"Jim Cramer on Fiserv, Inc. (FI): ‘Now It Was A...",https://news.google.com/rss/articles/CBMixAFBV...,NaN,FI,NaN


# Filter Irrelevant News

In [ ]:
all_news_cleaned_df = pd.DataFrame()

for ticker in tqdm(tickers_sorted):
    news_temp = all_news_df[all_news_df["Ticker"] == ticker].copy()
    news_temp = remove_irrelevant_news(news_temp.copy(), "title", threshold=0.1, threshold_size=0.1)

    all_news_cleaned_df = pd.concat([all_news_cleaned_df, news_temp.copy()])

all_news_cleaned_df = all_news_cleaned_df.reset_index(drop=True)

100%|██████████| 498/498 [01:24<00:00,  5.90it/s]


In [86]:
all_news_cleaned_df.shape

(472249, 6)

# Train Test Split

In [88]:
all_news_cleaned_df["date"] = pd.to_datetime(all_news_cleaned_df["date"], infer_datetime_format=True)
news_train = all_news_cleaned_df[all_news_cleaned_df["date"].dt.year < 2024].copy()
news_test = all_news_cleaned_df[all_news_cleaned_df["date"].dt.year >= 2024].copy()

In [91]:
print(news_train.shape)
print(news_test.shape)

# Removing Similar News


In [100]:
news_train_filtered = pd.DataFrame()
news_test_filtered = pd.DataFrame()

threshold = 0.75
for ticker in tqdm(tickers_sorted):
    train_temp = news_train[news_train["Ticker"] == ticker].copy()
    train_temp = remove_similar_news(train_temp, "title", threshold=threshold).reset_index(drop=True)
    news_train_filtered = pd.concat([news_train_filtered, train_temp.copy()])

    test_temp = news_test[news_test["Ticker"] == ticker].copy()
    all_test_dates = test_temp['date'].unique()
    test_news_filtered_temp = pd.DataFrame()
    for d in all_test_dates:
        test_temp2 = test_temp[test_temp['date']==d].copy()
        test_temp2 = remove_similar_news(test_temp2, "title", threshold=threshold)
        test_news_filtered_temp = pd.concat([test_news_filtered_temp, test_temp2])
    news_test_filtered = pd.concat([news_test_filtered, test_news_filtered_temp.copy()])


news_train_filtered = news_train_filtered.reset_index(drop=True)
news_test_filtered = news_test_filtered.reset_index(drop=True)

100%|██████████| 498/498 [03:36<00:00,  2.30it/s]


In [101]:
print(news_train_filtered.shape)
print(news_test_filtered.shape)

In [104]:
news_train_filtered.to_csv("Data/news_data_train.csv")
news_test_filtered.to_csv("Data/news_data_test.csv")

# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [4]:
stocks = yf.download(tickers, threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-01)')
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


In [5]:
stocks

Ticker           FTNT                                                        \
Price            Open       High        Low      Close  Adj Close    Volume   
Date                                                                          
2014-01-02   3.832000   3.846000   3.790000   3.804000   3.804000  13465000   
2014-01-03   3.844000   3.874000   3.786000   3.854000   3.854000  11757000   
2014-01-06   3.884000   3.944000   3.844000   3.848000   3.848000  24980500   
2014-01-07   3.864000   3.978000   3.836000   3.966000   3.966000  13815500   
2014-01-08   3.982000   4.010000   3.932000   3.968000   3.968000  10703000   
...               ...        ...        ...        ...        ...       ...   
2024-11-22  94.320000  94.500000  92.330002  92.769997  92.769997   4060500   
2024-11-25  93.110001  94.900002  92.680000  93.120003  93.120003   8804500   
2024-11-26  93.970001  96.699997  93.970001  96.440002  96.440002   4729300   
2024-11-27  96.680000  96.790001  93.860001  94.059998  94.059998   3834700   
2024-11-29  94.330002  95.470001  94.110001  95.050003  95.050003   2124100   

Ticker           SWKS                                   ...         IP  \
Price            Open       High        Low      Close  ...        Low   
Date                                                    ...              
2014-01-02  28.299999  28.340000  27.200001  27.400000  ...  45.331272   
2014-01-03  27.400000  27.750000  27.400000  27.719999  ...  45.265900   
2014-01-06  27.870001  27.930000  27.350000  27.629999  ...  45.265900   
2014-01-07  27.650000  28.000000  27.610001  27.790001  ...  45.321934   
2014-01-08  27.950001  28.040001  27.760000  27.920000  ...  45.321934   
...               ...        ...        ...        ...  ...        ...   
2024-11-22  85.000000  85.570000  84.720001  85.410004  ...  58.570000   
2024-11-25  86.550003  89.070000  86.070000  87.930000  ...  59.169998   
2024-11-26  88.120003  88.449997  85.720001  86.279999  ...  58.580002   
2024-11-27  86.279999  87.019997  85.139999  86.800003  ...  58.150002   
2024-11-29  87.199997  88.639999  87.180000  87.589996  ...  57.959999   

Ticker                                           HRL                        \
Price           Close  Adj Close   Volume       Open       High        Low   
Date                                                                         
2014-01-02  45.574085  28.950808  2305077  22.555000  22.584999  22.315001   
2014-01-03  45.405983  28.844023  2490965  22.495001  22.559999  22.320000   
2014-01-06  45.284576  28.766907  2436569  22.545000  22.545000  22.305000   
2014-01-07  45.723507  29.045732  3005477  22.430000  22.745001  22.430000   
2014-01-08  45.770203  29.075382  2569775  22.575001  22.580000  22.305000   
...               ...        ...      ...        ...        ...        ...   
2024-11-22  59.320000  59.320000  3454800  30.559999  30.870001  30.520000   
2024-11-25  59.570000  59.570000  6833600  30.980000  31.459999  30.950001   
2024-11-26  59.110001  59.110001  4450000  31.360001  31.709999  31.230000   
2024-11-27  58.380001  58.380001  2350800  31.780001  32.060001  31.680000   
2024-11-29  58.830002  58.830002  1819800  31.900000  32.490002  31.850000   

Ticker                                     
Price           Close  Adj Close   Volume  
Date                                       
2014-01-02  22.315001  17.616207  1372000  
2014-01-03  22.455000  17.726730  1358200  
2014-01-06  22.395000  17.679358  2499000  
2014-01-07  22.645000  17.876720  1192600  
2014-01-08  22.400000  17.683298  1320400  
...               ...        ...      ...  
2024-11-22  30.670000  30.670000  1999900  
2024-11-25  31.430000  31.430000  4316800  
2024-11-26  31.620001  31.620001  2312200  
2024-11-27  31.920000  31.920000  2220600  
2024-11-29  32.430000  32.430000  1904800  

[2747 rows x 3018 columns]

In [6]:
ticker_tz_df = pd.DataFrame()
for t in tickers:
    try:
        ticker_info = yf.Ticker(t)
        ticker_tz = ticker_info.info['timeZoneShortName']
        ticker_tz_temp_df = pd.DataFrame({'Ticker':[t], 'tz':[ticker_tz]})
        ticker_tz_df = pd.concat([ticker_tz_df, ticker_tz_temp_df])
    except:
        print(f'No timezone info for {t}')

No timezone info for BRK.B


In [7]:
print(f'Timezones of SPY stocks include: {ticker_tz_df.tz.unique()}')

Timezones of SPY stocks include: ['EST']


In [8]:
market = yf.download(["SPY"], threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [9]:
market

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-01-02,183.979996,184.070007,182.479996,182.919998,151.242935,119636900
2014-01-03,183.229996,183.600006,182.630005,182.889999,151.218140,81390600
2014-01-06,183.490005,183.559998,182.080002,182.360001,150.779877,108028200
2014-01-07,183.089996,183.789993,182.949997,183.479996,151.705933,86144200
2014-01-08,183.449997,183.830002,182.889999,183.520004,151.739014,96582300
...,...,...,...,...,...,...
2024-11-22,593.659973,596.150024,593.150024,595.510010,595.510010,38226400
2024-11-25,599.520020,600.859985,595.200012,597.530029,597.530029,42441400
2024-11-26,598.799988,601.330017,598.070007,600.650024,600.650024,45621300


In [10]:
stocks.to_excel("Data/stocks_data.xlsx")

In [11]:
market.to_excel("Data/market_data.xlsx")